<a href="https://colab.research.google.com/github/vandit-11/learning-data-science/blob/main/reranking_model_lightgbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LightGBM Personalized Re-Ranker

**Complete solution with:**
- ✅ **Sentence Transformer embeddings** for brand and category
- ✅ **Time-based weighting** (recent events weighted higher)
- ✅ **Labels 0-7** support ('added_to_PO' THEN 7, 'po_item_created' THEN 6, 'product_added' THEN 5, 'add_to_po_initiated' THEN 4, 'add_product_to_my_shop' THEN 3, 'quantity_modal_open' THEN 2, 'order_alarm_product_viewed' THEN 1, 'product_viewed' THEN 0)
- ✅ **Model persistence** with joblib (no retraining)
- ✅ **last90DaysEvents.csv** data

## Workflow:
1. **First Run**: Trains model, creates embeddings, saves everything
2. **Subsequent Runs**: Loads saved model instantly
3. **Testing**: Interactive testing with any user_id and item_ids


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from lightgbm import LGBMRanker
from sklearn.model_selection import GroupShuffleSplit
from sentence_transformers import SentenceTransformer
import joblib
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")


✓ Libraries imported successfully!


In [ ]:
# Check if model and metadata exist
MODEL_PATH = 'saved_model.pkl'
METADATA_PATH = 'model_metadata.pkl'

model_exists = os.path.exists(MODEL_PATH) and os.path.exists(METADATA_PATH)

if model_exists:
    print("✓ Trained model found! Will load saved model.")
else:
    print("⚠ No trained model found. Will train a new model.")

print(f"\nModel exists: {model_exists}")


⚠ No trained model found. Will train a new model.

Model exists: False


In [ ]:
### --- DEFINITIVE LOADING, CLEANING, AND SETUP BLOCK --- ###

print("--- Loading, Cleaning, and Setting Up DataFrames ---")

# 1. Load Data
print("\n[1/3] Loading data files...")
events_df = pd.read_csv('inputData/last90DaysEvents.csv')
buyer_features_df = pd.read_csv('inputData/buyerFeatures.csv')
item_features_df = pd.read_csv('inputData/itemFeatures.csv')
print("✓ Data loaded.")

# 2. Clean Data (Whitespace, Data Types, and Messy Labels)
print("\n[2/3] Cleaning data...")
# Clean buyer features
if 'user_id' in buyer_features_df.columns:
    buyer_features_df['user_id'] = buyer_features_df['user_id'].astype(str).str.strip()
else:
    # If it's already an index, clean the index itself
    buyer_features_df.index = buyer_features_df.index.astype(str).str.strip()

# Clean item features
if 'item_id' in item_features_df.columns:
    item_features_df['item_id'] = item_features_df['item_id'].astype(str).str.strip()
else:
    # If it's already an index, clean the index itself
    item_features_df.index = item_features_df.index.astype(str).str.strip()

# THE KEY FIX for your display issue: Clean the item_label column
if 'item_label' in item_features_df.columns:
    print("  -> Cleaning 'item_label' column for better display...")
    # This splits on newlines and takes the last part, fixing the "item_id\n..." issue
    item_features_df['item_label'] = item_features_df['item_label'].astype(str).str.split('\n').str[-1].str.strip()
print("✓ Data cleaned.")

# 3. Set Indices (Idempotent and Type-Safe)
print("\n[3/3] Setting DataFrame indices for fast lookups...")
if buyer_features_df.index.name != 'user_id':
    buyer_features_df.set_index('user_id', inplace=True)
    print("✓ Buyer features index set to 'user_id'.")
else:
    print("✓ Buyer features index is already 'user_id'.")

if item_features_df.index.name != 'item_id':
    # Before setting index, remove any duplicate item_ids to prevent errors
    if item_features_df['item_id'].duplicated().any():
        print("  -> Found and removed duplicate item_ids.")
        item_features_df.drop_duplicates(subset=['item_id'], keep='first', inplace=True)
    item_features_df.set_index('item_id', inplace=True)
    print("✓ Item features index set to 'item_id'.")
else:
    print("✓ Item features index is already 'item_id'.")

print("\n--- Setup Complete ---")

print(f"✓ Loaded {len(events_df):,} events")
print(f"✓ Loaded {len(buyer_features_df):,} buyer profiles")
print(f"✓ Loaded {len(item_features_df):,} items")
print(f"✓ Label range in events: {events_df['label'].min()} to {events_df['label'].max()}")

--- Loading, Cleaning, and Setting Up DataFrames ---

[1/3] Loading data files...
✓ Data loaded.

[2/3] Cleaning data...
  -> Cleaning 'item_label' column for better display...
✓ Data cleaned.

[3/3] Setting DataFrame indices for fast lookups...
✓ Buyer features index set to 'user_id'.
  -> Found and removed duplicate item_ids.
✓ Item features index set to 'item_id'.

--- Setup Complete ---
✓ Loaded 949,199 events
✓ Loaded 868,822 buyer profiles
✓ Loaded 1,187,563 items
✓ Label range in events: 0 to 7


In [ ]:
print(f"Original events count: {len(events_df)}")

# This ensures that when we drop duplicates, we keep the highest-value event.
events_df_sorted = events_df.sort_values('label', ascending=False)

# Drop duplicates based on user and item, keeping the first one (which is now the highest label)
events_df_deduplicated = events_df_sorted.drop_duplicates(subset=['user_id', 'item_id'], keep='first')

print(f"Deduplicated events count: {len(events_df_deduplicated)}")

events_df = events_df_deduplicated

Original events count: 949199
Deduplicated events count: 310552


In [ ]:
if not model_exists:
    print("="*80)
    print("TRAINING NEW MODEL WITH EMBEDDINGS & TIME WEIGHTING")
    print("="*80)

    # [1/8] Merge dataframes
    print("\n[1/8] Merging datasets...")
    merged_df = events_df.merge(buyer_features_df, on='user_id', how='left')
    merged_df = merged_df.merge(item_features_df, on='item_id', how='left')
    print(f"✓ Merged dataset: {merged_df.shape}")

    # [2/8] Create embeddings for brand_label and category_label
    print("\n[2/8] Creating embeddings for brand_label and category_label...")

    # Check if embeddings already exist
    if os.path.exists('brand_embeddings.pkl') and os.path.exists('category_embeddings.pkl'):
        print("  Loading cached embeddings...")
        brand_embedding_dict = joblib.load('brand_embeddings.pkl')
        category_embedding_dict = joblib.load('category_embeddings.pkl')
    else:
        print("  Generating embeddings with all-MiniLM-L6-v2...")
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

        # Build a comprehensive vocabulary from both item and user features
        print("  Building comprehensive vocabulary from item and user features...")
        all_brands = pd.concat([
            item_features_df['brand_label'],
            buyer_features_df['user_favorite_brand_label']
        ]).fillna('unknown').unique()

        all_categories = pd.concat([
            item_features_df['category_label'],
            buyer_features_df['user_favorite_category_label']
        ]).fillna('unknown').unique()

        print(f"  Encoding {len(all_brands):,} unique brands...")
        brand_embeddings = embedding_model.encode(all_brands.tolist(), show_progress_bar=True)
        brand_embedding_dict = dict(zip(all_brands, brand_embeddings))
        if 'unknown' not in brand_embedding_dict and len(brand_embeddings) > 0:
             brand_embedding_dict['unknown'] = np.zeros_like(brand_embeddings[0])

        print(f"  Encoding {len(all_categories):,} unique categories...")
        category_embeddings = embedding_model.encode(all_categories.tolist(), show_progress_bar=True)
        category_embedding_dict = dict(zip(all_categories, category_embeddings))
        if 'unknown' not in category_embedding_dict and len(category_embeddings) > 0:
             category_embedding_dict['unknown'] = np.zeros_like(category_embeddings[0])

        # Save embeddings
        joblib.dump(brand_embedding_dict, 'brand_embeddings.pkl')
        joblib.dump(category_embedding_dict, 'category_embeddings.pkl')
        print("  ✓ Embeddings cached to disk")

    # [3/8] Add embeddings to merged_df
    print("\n[3/8] Adding embeddings to dataset...")
    # Handle NaNs in all relevant label columns before creating embeddings
    merged_df['brand_label'].fillna('unknown', inplace=True)
    merged_df['category_label'].fillna('unknown', inplace=True)
    merged_df['user_favorite_brand_label'].fillna('unknown', inplace=True)
    merged_df['user_favorite_category_label'].fillna('unknown', inplace=True)

    # Get embedding dimension
    emb_dim = len(list(brand_embedding_dict.values())[0])
    print(f"  Embedding dimension: {emb_dim}")

    ### MODIFIED: Renamed item embedding columns to match your downstream code
    # Add item brand and category embeddings (prefixed with 'item_')
    brand_emb_list = [brand_embedding_dict.get(brand, brand_embedding_dict['unknown']) for brand in merged_df['brand_label']]
    for i in range(emb_dim):
        # This now creates columns like 'brand_emb_0', 'brand_emb_1', etc.
        merged_df[f'brand_emb_{i}'] = [emb[i] for emb in brand_emb_list]

    category_emb_list = [category_embedding_dict.get(cat, category_embedding_dict['unknown']) for cat in merged_df['category_label']]
    for i in range(emb_dim):
        # This now creates columns like 'category_emb_0', 'category_emb_1', etc.
        merged_df[f'category_emb_{i}'] = [emb[i] for emb in category_emb_list]

    print(f"  ✓ Added {emb_dim} item brand embedding features (as 'brand_emb_...')")
    print(f"  ✓ Added {emb_dim} item category embedding features (as 'category_emb_...')")

    # Apply embeddings to user favorite brand/category (prefixed with 'user_fav_')
    print("\n  Applying embeddings to user favorite brand/category...")
    # Add user favorite brand embeddings
    user_fav_brand_emb_list = [brand_embedding_dict.get(brand, brand_embedding_dict['unknown']) for brand in merged_df['user_favorite_brand_label']]
    for i in range(emb_dim):
        merged_df[f'user_fav_brand_emb_{i}'] = [emb[i] for emb in user_fav_brand_emb_list]

    # Add user favorite category embeddings
    user_fav_cat_emb_list = [category_embedding_dict.get(cat, category_embedding_dict['unknown']) for cat in merged_df['user_favorite_category_label']]
    for i in range(emb_dim):
        merged_df[f'user_fav_cat_emb_{i}'] = [emb[i] for emb in user_fav_cat_emb_list]

    print(f"  ✓ Added {emb_dim} user favorite brand embedding features")
    print(f"  ✓ Added {emb_dim} user favorite category embedding features")


    # [4/8] Add time-based weighting
    print("\n[4/8] Adding time-based weighting...")
    merged_df['eventTriggerTime'] = pd.to_datetime(merged_df['eventTriggerTime'], format='mixed', utc=True)
    max_date = merged_df['eventTriggerTime'].max()
    min_date = merged_df['eventTriggerTime'].min()
    merged_df['days_from_end'] = (max_date - merged_df['eventTriggerTime']).dt.days
    merged_df['time_weight'] = 1 / (1 + merged_df['days_from_end'] / 30)
    print(f"  ✓ Time range: {min_date.date()} to {max_date.date()}")
    print(f"  ✓ Time weight range: {merged_df['time_weight'].min():.3f} to {merged_df['time_weight'].max():.3f}")


    # [5/8] Handle missing values
    print("\n[5/8] Handling missing values...")
    numerical_cols = merged_df.select_dtypes(include=[np.number]).columns
    merged_df[numerical_cols] = merged_df[numerical_cols].fillna(0)
    categorical_cols = merged_df.select_dtypes(include=['object']).columns
    categorical_cols = [col for col in categorical_cols if col not in ['user_id', 'item_id', 'eventTriggerTime']]
    merged_df[categorical_cols] = merged_df[categorical_cols].fillna('unknown')
    print("  ✓ Missing values handled")


    # [6/8] Feature engineering
    print("\n[6/8] Feature engineering...")
    exclude_columns = ['user_id', 'item_id', 'item_label', 'label', 'eventTriggerTime',
                       'brand_label', 'category_label', 'days_from_end',
                       'user_favorite_brand_label', 'user_favorite_category_label']
    id_columns = [col for col in merged_df.columns if col.endswith('_id')]
    exclude_columns.extend(id_columns)
    feature_columns = [col for col in merged_df.columns if col not in exclude_columns]
    categorical_features = [col for col in feature_columns if merged_df[col].dtype == 'object']
    for col in categorical_features:
        merged_df[col] = merged_df[col].astype('category')
    print(f"  ✓ Total features: {len(feature_columns)}")
    print(f"  ✓ Categorical: {len(categorical_features)}")
    print(f"  ✓ Numerical: {len(feature_columns) - len(categorical_features)}")
    print(f"  ✓ Includes: {emb_dim*4} embedding dims + time_weight")


    # [7/8] Train/test split
    print("\n[7/8] Creating train/test split (80/20)...")
    merged_df_sorted = merged_df.sort_values('user_id').reset_index(drop=True)

    X = merged_df_sorted[feature_columns]
    y = merged_df_sorted['label'].astype(int)  # Labels are 0-7
    groups = merged_df_sorted['user_id']
    time_weights = merged_df_sorted['time_weight'].values  # For sample weighting

    # Split by user groups
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups.iloc[train_idx]
    groups_test = groups.iloc[test_idx]
    sample_weights_train = time_weights[train_idx]

    # Create group arrays
    train_group_sizes = groups_train.value_counts().sort_index()
    test_group_sizes = groups_test.value_counts().sort_index()
    train_groups = train_group_sizes.values
    test_groups = test_group_sizes.values

    print(f"  ✓ Training: {len(X_train):,} samples from {len(train_groups):,} users")
    print(f"  ✓ Testing:  {len(X_test):,} samples from {len(test_groups):,} users")
    print(f"  ✓ Label range: {y.min()} to {y.max()}")

    # [8/8] Train model with time-based sample weighting
    print("\n[8/8] Training LightGBM Ranker...")
    print("  Using time-based sample weights (recent events weighted higher)")

    ranker = LGBMRanker(
        objective='lambdarank',
        metric='ndcg',
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=31,
        max_depth=-1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    # Train with sample weights
    ranker.fit(X_train, y_train, group=train_groups, sample_weight=sample_weights_train)
    print("  ✓ Model training completed!")

    # Evaluate
    print("\n  Evaluating model on test set...")

    def calculate_ndcg_at_k(y_true, y_pred, groups, k=10):
        ndcg_scores = []
        start_idx = 0
        for group_size in groups:
            end_idx = start_idx + group_size
            y_true_group = y_true.iloc[start_idx:end_idx].values
            y_pred_group = y_pred[start_idx:end_idx]

            sorted_indices = np.argsort(y_pred_group)[::-1][:k]
            dcg = sum((2 ** y_true_group[idx] - 1) / np.log2(i + 2)
                      for i, idx in enumerate(sorted_indices))

            ideal_sorted = np.sort(y_true_group)[::-1][:k]
            idcg = sum((2 ** rel - 1) / np.log2(i + 2)
                       for i, rel in enumerate(ideal_sorted))

            if idcg > 0:
                ndcg_scores.append(dcg / idcg)
            start_idx = end_idx
        return np.mean(ndcg_scores)

    def calculate_precision_at_k(y_true, y_pred, groups, k=5):
        precision_scores = []
        start_idx = 0
        for group_size in groups:
            end_idx = start_idx + group_size
            y_true_group = y_true.iloc[start_idx:end_idx].values
            y_pred_group = y_pred[start_idx:end_idx]

            if group_size >= k:
                top_k_indices = np.argsort(y_pred_group)[::-1][:k]
                relevant_in_top_k = np.sum(y_true_group[top_k_indices] > 0)
                precision_scores.append(relevant_in_top_k / k)
            start_idx = end_idx
        return np.mean(precision_scores)

    # Get predictions
    train_pred = ranker.predict(X_train)
    test_pred = ranker.predict(X_test)

    # Calculate metrics
    train_ndcg_10 = calculate_ndcg_at_k(y_train, train_pred, train_groups, k=10)
    test_ndcg_10 = calculate_ndcg_at_k(y_test, test_pred, test_groups, k=10)
    train_p5 = calculate_precision_at_k(y_train, train_pred, train_groups, k=5)
    test_p5 = calculate_precision_at_k(y_test, test_pred, test_groups, k=5)

    print("\n" + "="*80)
    print("MODEL PERFORMANCE METRICS")
    print("="*80)
    print(f"{'Metric':<30} {'Training':<15} {'Testing':<15}")
    print("-"*80)
    print(f"{'NDCG@10':<30} {train_ndcg_10:.4f}          {test_ndcg_10:.4f}")
    print(f"{'Precision@5':<30} {train_p5:.4f}          {test_p5:.4f}")
    print("-"*80)
    print(f"\n✓ Model Performance: Test NDCG@10 = {test_ndcg_10:.2%}")

    # Save model and metadata
    print("\nSaving model, embeddings, and metadata...")

    metadata = {
        'feature_columns': feature_columns,
        'categorical_features': categorical_features,
        'test_ndcg_10': test_ndcg_10,
        'brand_embedding_dict': brand_embedding_dict,
        'category_embedding_dict': category_embedding_dict,
        'embedding_dim': emb_dim
    }

    joblib.dump(ranker, MODEL_PATH)
    joblib.dump(metadata, METADATA_PATH)

    print(f"✓ Model saved: {MODEL_PATH}")
    print(f"✓ Metadata saved: {METADATA_PATH}")
    print("\n" + "="*80)
    print("TRAINING COMPLETE!")
    print("="*80)

else:
    print("\n✓ Skipping training - loading saved model instead")


TRAINING NEW MODEL WITH EMBEDDINGS & TIME WEIGHTING

[1/8] Merging datasets...
✓ Merged dataset: (310552, 22)

[2/8] Creating embeddings for brand_label and category_label...
  Loading cached embeddings...

[3/8] Adding embeddings to dataset...
  Embedding dimension: 384
  ✓ Added 384 item brand embedding features (as 'brand_emb_...')
  ✓ Added 384 item category embedding features (as 'category_emb_...')

  Applying embeddings to user favorite brand/category...
  ✓ Added 384 user favorite brand embedding features
  ✓ Added 384 user favorite category embedding features

[4/8] Adding time-based weighting...
  ✓ Time range: 2025-08-08 to 2025-11-07
  ✓ Time weight range: 0.250 to 1.000

[5/8] Handling missing values...
  ✓ Missing values handled

[6/8] Feature engineering...
  ✓ Total features: 1546
  ✓ Categorical: 2
  ✓ Numerical: 1544
  ✓ Includes: 1536 embedding dims + time_weight

[7/8] Creating train/test split (80/20)...
  ✓ Training: 249,784 samples from 54,805 users
  ✓ Testing: 

In [ ]:
# Filter rows where brand_label is not 'unknown'
brand_known_df = merged_df[merged_df['brand_label'] != 'unknown']

# Select only embedding columns
brand_emb_cols = [col for col in merged_df.columns if col.startswith('brand_emb_')]

# Display brand label + embedding columns
print("🔹 Brand Embeddings (only where brand_label is known):\n")
print(brand_known_df[['brand_label'] + brand_emb_cols].head(10))


🔹 Brand Embeddings (only where brand_label is known):

                 brand_label  brand_emb_0  brand_emb_1  brand_emb_2  \
0                    Nilon's    -0.068148     0.047274    -0.022482   
1       Milkomore Pashu Ahar     0.012730     0.089231    -0.049366   
2  Non Marka agri cattlefeed    -0.026318    -0.054980    -0.018916   
3  Non Marka agri cattlefeed    -0.026318    -0.054980    -0.018916   
4       Milkomore Pashu Ahar     0.012730     0.089231    -0.049366   
5  Non Marka agri cattlefeed    -0.026318    -0.054980    -0.018916   
6       Milkomore Pashu Ahar     0.012730     0.089231    -0.049366   
7       Milkomore Pashu Ahar     0.012730     0.089231    -0.049366   
8  Non Marka agri cattlefeed    -0.026318    -0.054980    -0.018916   
9                   BELITAAS    -0.019437     0.020990    -0.043329   

   brand_emb_3  brand_emb_4  brand_emb_5  brand_emb_6  brand_emb_7  \
0     0.005238    -0.069531    -0.101483     0.067588    -0.048563   
1    -0.012695    -0.06

In [ ]:
print("Loading saved model and embeddings...")

# Load model and metadata
ranker = joblib.load(MODEL_PATH)
metadata = joblib.load(METADATA_PATH)

feature_columns = metadata['feature_columns']
categorical_features = metadata['categorical_features']
test_ndcg_10 = metadata['test_ndcg_10']
brand_embedding_dict = metadata['brand_embedding_dict']
category_embedding_dict = metadata['category_embedding_dict']
embedding_dim = metadata['embedding_dim']

print(f"✓ Model loaded!")
print(f"✓ Test NDCG@10: {test_ndcg_10:.4f} ({test_ndcg_10:.2%})")
print(f"✓ Features: {len(feature_columns)}")
print(f"✓ Brand embeddings: {len(brand_embedding_dict):,}")
print(f"✓ Category embeddings: {len(category_embedding_dict):,}")
print(f"✓ Embedding dimension: {embedding_dim}")


Loading saved model and embeddings...
✓ Model loaded!
✓ Test NDCG@10: 0.9205 (92.05%)
✓ Features: 1546
✓ Brand embeddings: 36,200
✓ Category embeddings: 310
✓ Embedding dimension: 384


## Step 5: Define Inference Function

This function uses embeddings for brand/category and combines search scores with personalization.


In [ ]:
### CORRECTED RERANKING FUNCTION (Final, Robust Version) ###

def normalize_scores(scores):
    """Normalize scores using Min-Max scaling"""
    scores = np.array(scores)
    min_score, max_score = np.min(scores), np.max(scores)
    if max_score == min_score:
        return np.ones_like(scores)
    return (scores - min_score) / (max_score - min_score)

def rerank_items(user_id, item_ids_list, original_scores_list, weight=0.2):
    """
    Re-rank items for a specific user using embeddings.
    NOTE: Assumes buyer_features_df and item_features_df have been indexed.
    """
    try:
        user_features = buyer_features_df.loc[user_id]
        if isinstance(user_features, pd.DataFrame): # Handle duplicate users
            user_features = user_features.iloc[0]
    except KeyError:
        print(f"Warning: User {user_id} not found. Using defaults.")
        user_features = pd.Series({
            col: 0 if buyer_features_df[col].dtype.kind in 'if' else 'unknown'
            for col in buyer_features_df.columns
        })

    user_fav_brand_label = user_features.get('user_favorite_brand_label', 'unknown')
    user_fav_category_label = user_features.get('user_favorite_category_label', 'unknown')
    user_fav_brand_emb = brand_embedding_dict.get(user_fav_brand_label, brand_embedding_dict.get('unknown'))
    user_fav_cat_emb = category_embedding_dict.get(user_fav_category_label, category_embedding_dict.get('unknown'))

    inference_data = []
    valid_items = []
    valid_scores = []

    for item_id, orig_score in zip(item_ids_list, original_scores_list):
        try:
            item_features = item_features_df.loc[item_id]
            ### ADDED: Robustness check for duplicate item_ids ###
            if isinstance(item_features, pd.DataFrame):
                # If for some reason duplicates still exist, just take the first one.
                item_features = item_features.iloc[0]
        except KeyError:
            print(f"Warning: Item {item_id} not found. Skipping.")
            continue

        combined = {}

        # 1. Add user features
        for col_name, value in user_features.items():
            combined[col_name] = value

        # 2. Add item features
        for col_name, value in item_features.items():
            combined[col_name] = value

        # 3. Create item embedding features
        item_brand_label = item_features.get('brand_label', 'unknown')
        item_category_label = item_features.get('category_label', 'unknown')
        item_brand_emb = brand_embedding_dict.get(item_brand_label, brand_embedding_dict.get('unknown'))
        item_cat_emb = category_embedding_dict.get(item_category_label, category_embedding_dict.get('unknown'))

        for i in range(emb_dim):
            combined[f'brand_emb_{i}'] = item_brand_emb[i]
            combined[f'category_emb_{i}'] = item_cat_emb[i]

        # 4. Create user favorite embedding features
        for i in range(emb_dim):
            combined[f'user_fav_brand_emb_{i}'] = user_fav_brand_emb[i]
            combined[f'user_fav_cat_emb_{i}'] = user_fav_cat_emb[i]

        combined['time_weight'] = 1.0
        inference_data.append(combined)
        valid_items.append(item_id)
        valid_scores.append(orig_score)

    if not inference_data:
        return item_ids_list

    inference_df = pd.DataFrame(inference_data)

    for col in feature_columns:
        if col not in inference_df.columns:
            inference_df[col] = 0

    for col in categorical_features:
        if col in inference_df.columns:
             train_categories = X_train[col].cat.categories
             inference_df[col] = pd.Categorical(inference_df[col], categories=train_categories).fillna('unknown')

    inference_df = inference_df[feature_columns]

    personalization_scores = ranker.predict(inference_df)
    norm_original = normalize_scores(valid_scores)
    norm_personalization = normalize_scores(personalization_scores)
    final_scores = (1 - weight) * norm_original + weight * norm_personalization
    ranked_items = sorted(zip(valid_items, final_scores), key=lambda x: x[1], reverse=True)

    return [item_id for item_id, _ in ranked_items]

print("✓ Inference function ready with embedding support!")
# print("\nUsage: rerank_items(user_id, item_ids_list, original_scores_list, weight=0.5)")

✓ Inference function ready with embedding support!


---
# Interactive Testing Section

## Test 1: High Personalization (weight=0.8)


In [ ]:
### CORRECTED TESTING AND DISPLAY BLOCK ###

# Test with specific user
user_id = "a9eca628-880f-4d0b-b29d-b219940dcc4f"

search_results = [
    "3aa89170-6606-49b6-8891-7808a2159335",
    "3574ebfc-4a26-47a6-b4f3-3691f0ff09d6",
    "4cfecfbb-83e3-422b-be55-b6ad5ca83959",
    "7fb96dca-f804-49f9-9a11-8ffff76e6715",
    "20a49af4-8590-4314-bfce-01f390178a42",
    "1b761fcd-4d2a-4d1b-ad5f-3e15b0a34b6e",
    "2d1f5249-5fec-45be-a055-91c960e80c14",
    "38d59699-fe61-46cf-a844-70c2aca95ffd",
    "2a336410-5dec-49cb-b972-3be48ce8b46a",
    "160344e7-3387-462e-82ea-ba8e28d9efc9"
]

search_scores = [100, 95, 90, 85, 80, 75, 70, 65, 60, 55]

print(f"User ID: {user_id}")
print(f"Items to rank: {len(search_results)}")

# Apply re-ranking
personalized_results = rerank_items(
    user_id=user_id,
    item_ids_list=search_results,
    original_scores_list=search_scores,
    weight=0.5
)

print("\n✅ Reranked Results:")
for i, item_id in enumerate(personalized_results, 1):
    original_pos = search_results.index(item_id) + 1

    try:
        item_info = item_features_df.loc[item_id]

        ### MODIFIED LINE: Explicitly convert item_name to a string ###
        item_name = str(item_info['item_label'])[:50]

        rank_change = original_pos - i

        if rank_change > 0:
            change = f"↑+{rank_change}"
        elif rank_change < 0:
            change = f"↓{rank_change}"
        else:
            change = "→0"

        print(f"{i:2d}. [{change:>4s}] {item_name:50s} (was #{original_pos})")

    except KeyError:
        print(f"{i:2d}. {item_id} (Item info not found)")

User ID: a9eca628-880f-4d0b-b29d-b219940dcc4f
Items to rank: 10

✅ Reranked Results:
 1. [ ↑+3] Quinoa Curls, 50 Gram                              (was #4)
 2. [ ↑+1] Haldiram - Namak Para, 500 Gram                    (was #3)
 3. [ ↓-2] Haldiram - Kaju Katli, 250 Gram                    (was #1)
 4. [ ↑+6] Panne Pasta 70 Gram                                (was #10)
 5. [ ↓-3] Haldiram - Rose Raisin Burfi, 500 Gram             (was #2)
 6. [ ↑+3] Haldiram - Baked Madrashi & Achari Mathi Combo Wit (was #9)
 7. [  →0] Haldiram - Kadhi Pakoda With Plain Rice, 350 Gram  (was #7)
 8. [ ↓-3] Feni Lachha (Crispy Vermicelly), 250 Gram          (was #5)
 9. [ ↓-1] Haldiram - Ginger Bread Housewall, 100 Gram        (was #8)
10. [ ↓-4] Haldiram - Chocolate Fudge Assorted, 200 Gram      (was #6)


# Test-2

In [ ]:
# Test with a specific user
user_id = "053cd37c-0bf7-45b5-a8dd-4765d0397823"

search_results = [
    "80084ef5-0974-4e69-a799-b1e56abbed00",
    "af00a1b4-a59f-4469-babb-ac8a452c4081",
    "4ca1a63e-617f-44f4-815c-09d1e4f8b3c0",
    "606b5d64-007b-4ffd-a659-eb725cbd6106"
]

search_scores = [100, 95, 90, 85]

print(f"User ID: {user_id}")
print(f"Items to rank: {len(search_results)}")

# Apply re-ranking
personalized_results = rerank_items(
    user_id=user_id,
    item_ids_list=search_results,
    original_scores_list=search_scores,
    weight=1.0
)

print("\n✅ Reranked Results:")
for i, item_id in enumerate(personalized_results, 1):
    original_pos = search_results.index(item_id) + 1

    try:
        ### MODIFIED LINE: Use .loc for fast and correct index lookup ###
        # This returns a Series (a single row) of the item's data
        item_info = item_features_df.loc[item_id]

        # Now access the item_label directly from the Series
        item_name = str(item_info['item_label'])[:50]
        rank_change = original_pos - i

        if rank_change > 0:
            change = f"↑+{rank_change}"
        elif rank_change < 0:
            change = f"↓{rank_change}"
        else:
            change = "→0"

        print(f"{i:2d}. [{change:>4s}] {item_name:50s} (was #{original_pos})")

    except KeyError:
        # This will happen if an item_id from search_results is not in your item_features_df
        print(f"{i:2d}. {item_id} (Item info not found)")

User ID: 053cd37c-0bf7-45b5-a8dd-4765d0397823
Items to rank: 4

✅ Reranked Results:
 1. [ ↑+2] Garam Masala Pouch Rs. 5/-                         (was #3)
 2. [ ↑+2] Meat Masala 50 Gram                                (was #4)
 3. [ ↓-2] Brown Walnut kernels Kashmiri Akrot Giri 250 Gram  (was #1)
 4. [ ↓-2] White Walnut kernels Kashmiri Akrot Giri 250 Gram  (was #2)


In [ ]:
# Test with specific user
user_id = "053cd37c-0bf7-45b5-a8dd-4765d0397823"

search_results = [
    "4ca1a63e-617f-44f4-815c-09d1e4f8b3c0",
    "606b5d64-007b-4ffd-a659-eb725cbd6106",
    "80084ef5-0974-4e69-a799-b1e56abbed00",
    "af00a1b4-a59f-4469-babb-ac8a452c4081"
]

search_scores = [100, 90, 80, 70]

print(f"User ID: {user_id}")
print(f"Items to rank: {len(search_results)}")

# Apply re-ranking
personalized_results = rerank_items(
    user_id=user_id,
    item_ids_list=search_results,
    original_scores_list=search_scores,
    weight=0.5
)

print("\n✅ Reranked Results:")
for i, item_id in enumerate(personalized_results, 1):
    original_pos = search_results.index(item_id) + 1

    try:
        ### MODIFIED LINE: Use .loc for fast and correct index lookup ###
        # This returns a Series (a single row) of the item's data
        item_info = item_features_df.loc[item_id]

        # Now access the item_label directly from the Series
        item_name = str(item_info['item_label'])[:50]
        rank_change = original_pos - i

        if rank_change > 0:
            change = f"↑+{rank_change}"
        elif rank_change < 0:
            change = f"↓{rank_change}"
        else:
            change = "→0"

        print(f"{i:2d}. [{change:>4s}] {item_name:50s} (was #{original_pos})")

    except KeyError:
        # This will happen if an item_id from search_results is not in your item_features_df
        print(f"{i:2d}. {item_id} (Item info not found)")

User ID: 053cd37c-0bf7-45b5-a8dd-4765d0397823
Items to rank: 4

✅ Reranked Results:
 1. [  →0] Garam Masala Pouch Rs. 5/-                         (was #1)
 2. [  →0] Meat Masala 50 Gram                                (was #2)
 3. [  →0] Brown Walnut kernels Kashmiri Akrot Giri 250 Gram  (was #3)
 4. [  →0] White Walnut kernels Kashmiri Akrot Giri 250 Gram  (was #4)


# Test-3

In [ ]:
# Test with specific user
user_id = "593054a7-6cb1-489b-8aa3-4bca3cf44e3b"

search_results = [
    "3433f2ab-027e-4279-b5f9-57d4f8e092f3",
    "e0e93074-192e-4873-8f2d-991faa1b215e"
]

search_scores = [90, 80]

print(f"User ID: {user_id}")
print(f"Items to rank: {len(search_results)}")

# Apply re-ranking
personalized_results = rerank_items(
    user_id=user_id,
    item_ids_list=search_results,
    original_scores_list=search_scores,
    weight=0.5  # 50% personalization, 50% search
)

print("\n✅ Reranked Results:")
for i, item_id in enumerate(personalized_results, 1):
    original_pos = search_results.index(item_id) + 1

    try:
        ### MODIFIED LINE: Use .loc for fast and correct index lookup ###
        # This returns a Series (a single row) of the item's data
        item_info = item_features_df.loc[item_id]

        # Now access the item_label directly from the Series
        item_name = str(item_info['item_label'])[:50]
        rank_change = original_pos - i

        if rank_change > 0:
            change = f"↑+{rank_change}"
        elif rank_change < 0:
            change = f"↓{rank_change}"
        else:
            change = "→0"

        print(f"{i:2d}. [{change:>4s}] {item_name:50s} (was #{original_pos})")

    except KeyError:
        # This will happen if an item_id from search_results is not in your item_features_df
        print(f"{i:2d}. {item_id} (Item info not found)")

User ID: 593054a7-6cb1-489b-8aa3-4bca3cf44e3b
Items to rank: 2

✅ Reranked Results:
 1. [  →0] Tea 50 Gram                                        (was #1)
 2. [  →0] Tea 100 Gram                                       (was #2)


In [ ]:
# Test with specific user
user_id = "c341510b-5525-4409-b268-9e45d58ee26a"

search_results = [
    "c5f0cdc7-fe8e-4b12-b9fc-27bde0a66acd",
    "cc7a250f-ccce-4fc1-8330-786edb1bc460",
    "a1366d55-fda3-41e5-b76b-f64b7d5a56a1",
    "11d5f486-3d75-4303-ac06-fe8323274d7e",
    "15fe3354-2ab8-4adf-88ef-4a459d70cb8c",
    "37c47f6b-cd14-4a33-aaf6-8ed84fb06729"
]

search_scores = [90, 80, 70, 60, 50, 40]

print(f"User ID: {user_id}")
print(f"Items to rank: {len(search_results)}")

# Apply re-ranking
personalized_results = rerank_items(
    user_id=user_id,
    item_ids_list=search_results,
    original_scores_list=search_scores,
    weight=0.5  # 50% personalization, 50% search
)

print("\n✅ Reranked Results:")
for i, item_id in enumerate(personalized_results, 1):
    original_pos = search_results.index(item_id) + 1

    try:
        ### MODIFIED LINE: Use .loc for fast and correct index lookup ###
        # This returns a Series (a single row) of the item's data
        item_info = item_features_df.loc[item_id]

        # Now access the item_label directly from the Series
        item_name = str(item_info['item_label'])[:50]
        rank_change = original_pos - i

        if rank_change > 0:
            change = f"↑+{rank_change}"
        elif rank_change < 0:
            change = f"↓{rank_change}"
        else:
            change = "→0"

        print(f"{i:2d}. [{change:>4s}] {item_name:50s} (was #{original_pos})")

    except KeyError:
        # This will happen if an item_id from search_results is not in your item_features_df
        print(f"{i:2d}. {item_id} (Item info not found)")

User ID: c341510b-5525-4409-b268-9e45d58ee26a
Items to rank: 6

✅ Reranked Results:
 1. [  →0] Kamdhenu Chunni 50kg                               (was #1)
 2. [ ↑+2] Supreme Feed Dark Green 50 Kg                      (was #4)
 3. [ ↑+2] Supreme Feed Dark Green 25 Kg                      (was #5)
 4. [ ↓-2] Kalash Uttam Quality Chunni 50 Kg                  (was #2)
 5. [ ↓-2] Indorama Urja 5 kg                                 (was #3)
 6. [  →0] Indorama Urja 10 kg                                (was #6)


In [ ]:
# Test with specific user
user_id = "28ac6038-9d3d-4231-9ab2-85016970a3f7"

search_results = [
    "c5f0cdc7-fe8e-4b12-b9fc-27bde0a66acd",
    "cc7a250f-ccce-4fc1-8330-786edb1bc460",
    "a1366d55-fda3-41e5-b76b-f64b7d5a56a1",
    "11d5f486-3d75-4303-ac06-fe8323274d7e",
    "37c47f6b-cd14-4a33-aaf6-8ed84fb06729",
    "15fe3354-2ab8-4adf-88ef-4a459d70cb8c"
]

search_scores = [90, 80, 70, 60, 50, 40]

print(f"User ID: {user_id}")
print(f"Items to rank: {len(search_results)}")

# Apply re-ranking
personalized_results = rerank_items(
    user_id=user_id,
    item_ids_list=search_results,
    original_scores_list=search_scores,
    weight=0.5  # 50% personalization, 50% search
)

print("\n✅ Reranked Results:")
for i, item_id in enumerate(personalized_results, 1):
    original_pos = search_results.index(item_id) + 1

    try:
        ### MODIFIED LINE: Use .loc for fast and correct index lookup ###
        # This returns a Series (a single row) of the item's data
        item_info = item_features_df.loc[item_id]

        # Now access the item_label directly from the Series
        item_name = str(item_info['item_label'])[:50]
        rank_change = original_pos - i

        if rank_change > 0:
            change = f"↑+{rank_change}"
        elif rank_change < 0:
            change = f"↓{rank_change}"
        else:
            change = "→0"

        print(f"{i:2d}. [{change:>4s}] {item_name:50s} (was #{original_pos})")

    except KeyError:
        # This will happen if an item_id from search_results is not in your item_features_df
        print(f"{i:2d}. {item_id} (Item info not found)")

User ID: 28ac6038-9d3d-4231-9ab2-85016970a3f7
Items to rank: 6

✅ Reranked Results:
 1. [ ↑+1] Kalash Uttam Quality Chunni 50 Kg                  (was #2)
 2. [ ↓-1] Kamdhenu Chunni 50kg                               (was #1)
 3. [ ↑+3] Supreme Feed Dark Green 25 Kg                      (was #6)
 4. [  →0] Supreme Feed Dark Green 50 Kg                      (was #4)
 5. [ ↓-2] Indorama Urja 5 kg                                 (was #3)
 6. [ ↓-1] Indorama Urja 10 kg                                (was #5)


In [ ]:
events_df[events_df['user_id'] == "28ac6038-9d3d-4231-9ab2-85016970a3f7"]

,user_id,item_id,label,eventTriggerTime
160373,28ac6038-9d3d-4231-9ab2-85016970a3f7,15fe3354-2ab8-4adf-88ef-4a459d70cb8c,7,2025-09-21 14:08:40.817+05:30
160374,28ac6038-9d3d-4231-9ab2-85016970a3f7,87b23138-6858-483c-96ad-e094db0ab756,7,2025-09-21 14:09:51.794+05:30
102047,28ac6038-9d3d-4231-9ab2-85016970a3f7,054197fd-daea-4798-b649-e5370a2b2da9,7,2025-09-04 13:19:25.478+05:30


In [ ]:
merged_df

,user_id,item_id,label,eventTriggerTime,user_district,user_state,user_purchase_count,user_myshop_product_count,user_avg_purchase_price,user_favorite_category_id,...,user_fav_cat_emb_376,user_fav_cat_emb_377,user_fav_cat_emb_378,user_fav_cat_emb_379,user_fav_cat_emb_380,user_fav_cat_emb_381,user_fav_cat_emb_382,user_fav_cat_emb_383,days_from_end,time_weight
0,c217fb4f-4962-4cdb-b214-f400ecde5799,197a889b-b6d2-4300-916c-8b2b6ad8012e,7,2025-08-25 15:37:09.635000+00:00,Sheohar,Bihar,0.0,9.0,0.000000,unknown,...,-0.005578,-0.003543,-0.038568,0.010749,0.172103,-0.098167,-0.030852,0.024557,73,0.291262
1,c341510b-5525-4409-b268-9e45d58ee26a,15fe3354-2ab8-4adf-88ef-4a459d70cb8c,7,2025-09-21 08:42:12.507000+00:00,Amethi,Uttar Pradesh,56.0,17.0,8462.895813,7e8d9d89-5c69-4c5f-937b-39a854d77c40,...,-0.030047,-0.078701,0.055389,0.014256,-0.027818,0.034046,0.090709,-0.067537,47,0.389610
2,4578adb0-4a80-4319-b7f6-d48afa023e25,cc7a250f-ccce-4fc1-8330-786edb1bc460,7,2025-09-21 08:38:32.504000+00:00,Amethi,Uttar Pradesh,7.0,4.0,20767.919143,38db08a3-2f8e-44d5-9a72-6db5fcc38f4e,...,-0.070496,-0.109344,0.127150,-0.006921,-0.020120,-0.024873,0.060080,-0.043599,47,0.389610
3,4578adb0-4a80-4319-b7f6-d48afa023e25,c5f0cdc7-fe8e-4b12-b9fc-27bde0a66acd,7,2025-09-21 08:38:47.097000+00:00,Amethi,Uttar Pradesh,7.0,4.0,20767.919143,38db08a3-2f8e-44d5-9a72-6db5fcc38f4e,...,-0.070496,-0.109344,0.127150,-0.006921,-0.020120,-0.024873,0.060080,-0.043599,47,0.389610
4,28ac6038-9d3d-4231-9ab2-85016970a3f7,15fe3354-2ab8-4adf-88ef-4a459d70cb8c,7,2025-09-21 08:38:40.817000+00:00,Maharajganj,Uttar Pradesh,12.0,15.0,26343.154000,38db08a3-2f8e-44d5-9a72-6db5fcc38f4e,...,-0.070496,-0.109344,0.127150,-0.006921,-0.020120,-0.024873,0.060080,-0.043599,47,0.389610
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310547,58111cd6-9f5c-4b55-9417-75f98b65fd23,692c0408-99a2-4520-9f4a-6b7206e3aa92,0,2025-10-26 10:27:58.398000+00:00,North Delhi,Delhi,0.0,9.0,0.000000,unknown,...,-0.005578,-0.003543,-0.038568,0.010749,0.172103,-0.098167,-0.030852,0.024557,12,0.714286
310548,4fed17ff-84ea-467e-8ab1-903e38e43f14,f6e7b854-e875-4505-8a29-04b16dc82d1a,0,2025-10-26 10:06:08.284000+00:00,North West Delhi,Delhi,0.0,0.0,0.000000,unknown,...,-0.005578,-0.003543,-0.038568,0.010749,0.172103,-0.098167,-0.030852,0.024557,12,0.714286
310549,da0c0828-7548-49e1-9593-17bbf1c6648a,b29e94aa-effb-4c6e-9362-e9f40bd6ed3d,0,2025-10-26 10:04:16.019000+00:00,Gwalior,Madhya Pradesh,0.0,1.0,0.000000,unknown,...,-0.005578,-0.003543,-0.038568,0.010749,0.172103,-0.098167,-0.030852,0.024557,12,0.714286
310550,da0c0828-7548-49e1-9593-17bbf1c6648a,23685cf2-4f4d-4f6b-b9fd-74f5a398d5df,0,2025-10-26 10:04:41.388000+00:00,Gwalior,Madhya Pradesh,0.0,1.0,0.000000,unknown,...,-0.005578,-0.003543,-0.038568,0.010749,0.172103,-0.098167,-0.030852,0.024557,12,0.714286


In [ ]:
merged_df[merged_df['user_id'] == "0000fa64-522e-475e-a915-2fa78844c664"]

,user_id,item_id,label,eventTriggerTime,user_district,user_state,user_purchase_count,user_myshop_product_count,user_avg_purchase_price,user_favorite_category_id,...,user_fav_cat_emb_376,user_fav_cat_emb_377,user_fav_cat_emb_378,user_fav_cat_emb_379,user_fav_cat_emb_380,user_fav_cat_emb_381,user_fav_cat_emb_382,user_fav_cat_emb_383,days_from_end,time_weight
